# Workshop 02 — SFT-LoRA on Qwen3.5-4B VLM for Phishing Detection (Vision)

**Optional companion to notebook 01.** This trains a vision-language model on screenshots of phishing pages — the same task, different modality.

## Why this exists

Real phishing detection often relies on **how a page looks**, not just its text. A typosquatted PayPal page is obvious from the rendered visual even when the HTML is obfuscated.

Qwen3.5-4B is a vision-language model. Its JumpStart ID is `huggingface-vlm-qwen3-5-4b` and it supports SFT (LoRA).

## Prerequisite

Run `prep/02_prep_image_dataset.ipynb` first to publish a multimodal dataset to HuggingFace.

## ⚠️ Caveat — VLM dataset format

SageMaker's `SFTTrainer` documentation we have shows the `prompt + completion` JSONL schema for **text** SFT. For vision models the dataset format may differ (typically a `messages` field with multimodal content). Confirm the latest schema in:

- https://docs.aws.amazon.com/sagemaker/latest/dg/model-customize-open-weight.html
- https://docs.aws.amazon.com/sagemaker/latest/dg/model-customize-open-weight-samples.html

If the docs require a different schema for VLMs, adjust the `to_sft_record(...)` function in section 3 accordingly. The rest of the notebook (DataSet → SFTTrainer → deploy) is identical to notebook 01.

## 0. Install / upgrade SDK

In [ ]:
%pip install -q --upgrade sagemaker datasets boto3 rich pillow
# After this cell, **restart the kernel**.

## 1. Configuration

In [ ]:
HF_DATASET_ID = "gt2026workshop/phreshphish-2k"
HF_CONFIG_NAME = "image"
# S3 output — leave S3_BUCKET = None to use the account's DEFAULT SageMaker bucket
# (sagemaker-<region>-<account-id>). Session cell resolves S3_OUTPUT_PATH.
S3_BUCKET = None
S3_PREFIX = "qwen3-5-4b-phish-vlm-sft"
ROLE_ARN = None  # auto-detected

BASE_MODEL = "huggingface-vlm-qwen3-5-4b"   # Qwen3.5-4B VLM
MODEL_PACKAGE_GROUP_NAME = "qwen3-5-4b-phish-vlm-sft"
EXPERIMENT_NAME = "qwen3-5-4b-phish-vlm-sft"
RUN_NAME = "workshop-vlm-run-01"

ACCEPT_EULA = True

## 2. SageMaker session

In [ ]:
import os, boto3
from sagemaker.core.helper.session_helper import Session
from rich import print as rprint

REGION = boto3.Session().region_name
sm_client = boto3.client("sagemaker", region_name=REGION)
sagemaker_session = Session(sagemaker_client=sm_client)

# Default SageMaker bucket (sagemaker-<region>-<acct>); override via S3_BUCKET above.
BUCKET = S3_BUCKET or sagemaker_session.default_bucket()
S3_OUTPUT_PATH = f"s3://{BUCKET}/{S3_PREFIX}/"


if ROLE_ARN is None:
    from sagemaker.core.helper.session_helper import get_execution_role
    ROLE_ARN = get_execution_role()

os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = f"https://mlflow.sagemaker.{REGION}.app.aws"
rprint({"region": REGION, "role": ROLE_ARN, "output": S3_OUTPUT_PATH})

## 3. Pull the prepared image dataset and stage for SageMaker

The HF dataset has columns: `image` (PIL), `prompt` (str), `completion` (str).

We:
1. Save each row's image to S3 as `s3://.../images/<sha256>.png`
2. Build a JSONL where each row references its S3 image URI alongside the text prompt + completion
3. Register that JSONL with the AI Registry

In [ ]:
import io, json, pathlib, hashlib
from datasets import load_dataset
from urllib.parse import urlparse

ds = load_dataset(HF_DATASET_ID, name=HF_CONFIG_NAME)
print(ds)
print("Sample row keys:", list(ds["train"][0].keys()))

In [ ]:
p = urlparse(S3_OUTPUT_PATH.rstrip("/"))
BUCKET = p.netloc
PREFIX = p.path.lstrip("/")
DATA_PREFIX = f"{PREFIX}/data"
IMG_PREFIX = f"{PREFIX}/images"

s3 = boto3.client("s3")
tmp = pathlib.Path("./_staged_vlm"); tmp.mkdir(exist_ok=True)

def to_sft_record(image_s3_uri: str, prompt: str, completion: str) -> dict:
    """VLM SFT record — image referenced by S3 URI.

    NOTE: confirm exact schema with the SageMaker open-weight customization docs;
    this is a reasonable default that mirrors common multimodal SFT formats.
    """
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": image_s3_uri},
                {"type": "text", "text": prompt},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": completion}]},
        ],
    }

In [ ]:
from tqdm.auto import tqdm

def upload_split(split_name: str):
    out_path = tmp / f"{split_name}.jsonl"
    rows = ds[split_name]
    n = 0
    with open(out_path, "w") as f:
        for r in tqdm(rows, desc=f"upload {split_name}"):
            img = r["image"]
            buf = io.BytesIO()
            img.save(buf, format="PNG")
            data = buf.getvalue()
            sha = r.get("sha256") or hashlib.sha256(data).hexdigest()
            key = f"{IMG_PREFIX}/{split_name}/{sha}.png"
            s3.put_object(Bucket=BUCKET, Key=key, Body=data, ContentType="image/png")
            uri = f"s3://{BUCKET}/{key}"
            f.write(json.dumps(to_sft_record(uri, r["prompt"], r["completion"])) + "\n")
            n += 1
    s3_uri = f"s3://{BUCKET}/{DATA_PREFIX}/{split_name}.jsonl"
    s3.upload_file(str(out_path), BUCKET, f"{DATA_PREFIX}/{split_name}.jsonl")
    print(f"{split_name}: {n} rows → {s3_uri}")
    return s3_uri

TRAIN_S3 = upload_split("train")
VAL_S3 = upload_split("validation")

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

train_dataset = DataSet.create(name="phreshphish-workshop-image-train", source=TRAIN_S3, wait=True)
val_dataset = DataSet.create(name="phreshphish-workshop-image-val", source=VAL_S3, wait=True)
rprint({"train_arn": train_dataset.arn, "val_arn": val_dataset.arn})

## 4. Model package group + trainer

In [ ]:
from sagemaker.core.resources import ModelPackageGroup

try:
    mpg = ModelPackageGroup.create(
        model_package_group_name=MODEL_PACKAGE_GROUP_NAME,
        model_package_group_description="Qwen3.5-4B VLM fine-tuned for phishing screenshots",
    )
    print("created", mpg.model_package_group_arn)
except Exception:
    mpg = ModelPackageGroup.get(model_package_group_name=MODEL_PACKAGE_GROUP_NAME)
    print("reusing", MODEL_PACKAGE_GROUP_NAME)

In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType
from rich.pretty import pprint

trainer = SFTTrainer(
    model=BASE_MODEL,
    training_type=TrainingType.LORA,
    model_package_group=mpg,
    training_dataset=train_dataset.arn,
    validation_dataset=val_dataset.arn,
    s3_output_path=S3_OUTPUT_PATH,
    sagemaker_session=sagemaker_session,
    accept_eula=ACCEPT_EULA,
    role=ROLE_ARN,
    mlflow_experiment_name=EXPERIMENT_NAME,
    mlflow_run_name=RUN_NAME,
)

print("Default hyperparameters for Qwen3.5-4B VLM + LoRA:")
pprint(trainer.hyperparameters.to_dict())

## 5. Train

In [ ]:
training_job = trainer.train(wait=True)
print("Job:", training_job.training_job_name, "|", training_job.training_job_status)

## 6. Deploy + smoke test

In [ ]:
import random
from sagemaker.serve import ModelBuilder

endpoint_name = f"qwen3-5-4b-vlm-phish-{random.randint(1000, 9999)}"
model_builder = ModelBuilder(model=trainer)
model_builder.build()
endpoint = model_builder.deploy(endpoint_name=endpoint_name)
print("Deployed:", endpoint_name)

In [ ]:
import json, base64, io, boto3
rt = boto3.client("sagemaker-runtime", region_name=REGION)

def predict_vlm(image_pil, prompt: str) -> str:
    buf = io.BytesIO(); image_pil.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    body = {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": f"data:image/png;base64,{img_b64}"},
                {"type": "text", "text": prompt},
            ]}
        ],
        "parameters": {"max_new_tokens": 5, "temperature": 0.0, "do_sample": False},
    }
    resp = rt.invoke_endpoint(EndpointName=endpoint_name, Body=json.dumps(body), ContentType="application/json")
    return json.loads(resp["Body"].read())

# Eval on first 10 held-out images
samples = ds["validation"].select(range(10))
for r in samples:
    out = predict_vlm(r["image"], r["prompt"])
    print(f"truth={r['completion']}  out={out}")

## 7. Clean up

```python
# sm_client.delete_endpoint(EndpointName=endpoint_name)
# sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
```